# LangChain — ReAct Agent 示例

**ReAct（Reasoning + Acting）** 是一种让 LLM 交替进行"推理"和"行动"的框架。

## ReAct 循环

```
Question: 用户的问题
Thought:  我需要思考如何解决这个问题...
Action:   Search
Action Input: 搜索关键词
Observation: 搜索返回的结果
Thought:  根据搜索结果，我还需要...
Action:   Calculator
Action Input: 数学表达式
Observation: 计算结果
Thought:  我现在知道最终答案了
Final Answer: 最终答案
```

**核心思想：** LLM 不是一次性给出答案，而是反复"思考→行动→观察"，直到收集到足够信息。

**工具：**
- **Search**：DuckDuckGo 搜索（免费）
- **Calculator**：numexpr 数学计算

**模型：** 通义千问 (Qwen) via DashScope API

In [1]:
import sys
!{sys.executable} -m pip install -q langchain langchain-classic langchain-core langchain-community langchain-openai ddgs numexpr python-dotenv


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: C:\Lisa\ai-model-study\venv\Scripts\python.exe -m pip install --upgrade pip


## 1. 导入环境变量 & 初始化大模型

In [2]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen-plus",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=0.5,
)

response = llm.invoke("你好，请用一句话介绍你自己。")
print(response.content)

你好！我是通义千问（Qwen），阿里巴巴集团旗下的超大规模语言模型，能够回答问题、创作文字（如写故事、公文、邮件、剧本等）、进行逻辑推理、编程，还能表达观点，甚至玩游戏，支持多种语言，致力于为你提供高效、智能的交互体验。


## 2. 设置工具

- **Search**：DuckDuckGo 搜索（免费替代 SerpAPI）
- **Calculator**：numexpr 数学计算（替代已废弃的 llm-math）

In [3]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_core.tools import Tool
import numexpr

wrapper = DuckDuckGoSearchAPIWrapper(region="cn-zh", max_results=5, backend="api")
search = DuckDuckGoSearchRun(api_wrapper=wrapper)

def safe_search(query: str) -> str:
    """DuckDuckGo search with retry and error handling."""
    import time
    for attempt in range(3):
        try:
            return search.run(query)
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                return f"搜索失败，请根据已有知识回答。错误: {e}"

def calculator(expression: str) -> str:
    """Evaluate a math expression using numexpr."""
    try:
        expression = expression.strip().replace("^", "**")
        local_dict = {"pi": 3.141592653589793, "e": 2.718281828459045}
        result = numexpr.evaluate(expression, local_dict=local_dict)
        return str(float(result))
    except Exception as e:
        return f"计算出错，请检查表达式格式: {e}"

tools = [
    Tool(name="Search", func=safe_search,
         description="用于搜索互联网上的实时信息。输入应为搜索关键词。"),
    Tool(name="Calculator", func=calculator,
         description="用于数学计算。输入应为数学表达式，如 '100 / 25' 或 '3.14 * 2'。"),
]

print(f"已创建 {len(tools)} 个工具:")
for t in tools:
    print(f"  - {t.name}: {t.description}")

已创建 2 个工具:
  - Search: 用于搜索互联网上的实时信息。输入应为搜索关键词。
  - Calculator: 用于数学计算。输入应为数学表达式，如 '100 / 25' 或 '3.14 * 2'。


## 3. 定义 ReAct Prompt 模板

这是 ReAct 的核心 — 通过 Prompt 告诉 LLM 使用 **Thought → Action → Observation** 的循环模式。

模板中的关键变量：
- `{tools}`: 可用工具的名称和描述（自动填充）
- `{tool_names}`: 工具名称列表（自动填充）
- `{input}`: 用户问题
- `{agent_scratchpad}`: 之前的推理和观察记录

In [4]:
from langchain_core.prompts import PromptTemplate

template = '''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}'''

prompt = PromptTemplate.from_template(template)

print("ReAct Prompt 模板已定义")
print(f"输入变量: {prompt.input_variables}")

ReAct Prompt 模板已定义
输入变量: ['agent_scratchpad', 'input', 'tool_names', 'tools']


## 4. 初始化 ReAct Agent & AgentExecutor

- `create_react_agent`: 将 LLM + Tools + Prompt 组装成 ReAct agent
- `AgentExecutor`: 执行引擎，负责运行 Thought/Action/Observation 循环

In [6]:
from langchain_classic.agents import create_react_agent, AgentExecutor

agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    handle_parsing_errors=True,
    verbose=True,
    max_iterations=10,
    stream_runnable=False,
)

print("ReAct Agent 创建完成")

ReAct Agent 创建完成


## 5. 执行 AgentExecutor

观察 ReAct 循环：LLM 会交替进行 Thought → Action → Observation，直到得出最终答案。

In [8]:
result = agent_executor.invoke({"input": "地球到月球的距离是多少公里？光从地球到月球需要多少秒？"})
print("\n最终答案:", result["output"])



> Entering new AgentExecutor chain...
我需要查找中国市场上玫瑰花的当前进货价格，以便计算100元能买多少支。由于价格会因季节、产地、品质和采购渠道（如花卉批发市场 vs. 批发商）而波动，我应搜索最新的市场批发价或进货价信息。

Action: Search
Action Input: 中国玫瑰花批发价格 每支 元 2024年Job seekers are looking for suitable positions at a job fair for women in Fuyang, China, on March 6,2024.2025年3月4日. 35岁现象：中国互联网职场“人矿”的艰难求职故事.2024年8月15日. 上海封城下的外卖骑手 “这是挣的血汗钱”. 2024年起，adidas将根据最新的商品尺码命名规则，调整线上店铺所有商品的尺码介绍信息，陆续去除商品实物标签及商品详情页面尺码表中尺码旁的“A”或“J”. 标识，即不再在尺码中体现商品属于Asia版型或Japan版型。 苏州晴天被金主调教视频7. 在车内捧着鲜花. 晴天坐在金主车内，手捧一束红白玫瑰花束，眼神温柔注视着花朵，那喜爱之情显而易见，仿佛这礼物代表了金主的宠溺与占有。 苏州晴天被金主调教视频1. 苏州晴天被金主调教视频2. AI脱衣魔改国产电视剧玫瑰的故事 无数人的梦中女神刘亦菲被黑屌爆操侵犯 大量颜射！ 赛依依 • 2026年02 月 27 日 •每日大赛, 明星大赛. 知乎中国最大的问答社区.

APIError: Input data may contain inappropriate content. For details, see: https://help.aliyun.com/zh/model-studio/error-code#inappropriate-content

In [ ]:
result = agent_executor.invoke({"input": "2024年巴黎奥运会中国获得了多少金牌？平均每块金牌花费了多少天？"})
print("\n最终答案:", result["output"])

## 6. 纯计算问题

测试 Calculator 工具：Agent 应该识别出这是数学问题，直接调用计算器。

In [7]:
result = agent_executor.invoke({"input": "计算：如果一束玫瑰25元，买3束打8折，再加15元运费，总共多少钱？"})
print("\n最终答案:", result["output"])



> Entering new AgentExecutor chain...
需要计算总价：先算3束玫瑰的原价，再打8折（即乘以0.8），最后加上15元运费。  
步骤：  
1. 3束玫瑰原价 = 25 × 3  
2. 打8折后价格 = (25 × 3) × 0.8  
3. 加运费 = (25 × 3) × 0.8 + 15  

可用 Calculator 计算整个表达式。

Action: Calculator
Action Input: 25 * 3 * 0.8 + 1575.0Final Answer: 总共需要75元。

> Finished chain.

最终答案: 总共需要75元。
